In [95]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [319]:
class Value:
    def __init__(self, data, _children = (), _op = '', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.grad = 0.0
        self.label = label
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out =  Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other
        
    
    def tanh(self):
        x = self.data
        th = math.tanh(x)
        out = Value(th, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - (th)**2) * out.grad 
            
        out._backward = _backward
        return out


    def __truediv__(self, other):
        return self * other ** - 1

    
    def __rtruediv__(self, other):
        return self * other **-1
    
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
            
        out._backward = _backward
        return out

    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "int/float powers only!!"
        out = Value(self.data**other, (self, ), f'**{other}')
        def _backward():
            self.grad += (other * (self.data**(other-1))) * out.grad
        out._backward = _backward
        return out


    def __sub__(self, other):
        return self + (-other)

    
    def __rsub__(self, other):
        return self + (-other)
    
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        self.grad = 1
        for node in reversed(topo):
            node._backward()

In [97]:
from graphviz import Digraph


def trace(root):
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in getattr(v, '_prev', []):
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format="svg", graph_attr={"rankdir": "LR"})
    nodes, edges = trace(root)

    for n in nodes:
        uid = str(id(n))
        data_val = getattr(n, 'data', None)
        grad_val = getattr(n, 'grad', 0.0)
        try:
            data_str = f"{float(data_val):.4f}"
        except Exception:
            data_str = str(data_val)

        # Use str() instead of undefined string(), and fix label formatting
        dot.node(name=uid,
                 label="{ %s | data: %s | grad %.4f}" % (n.label, data_str, grad_val),
                 shape="record")

        # If this Value is the result of an operation, draw the op node
        if getattr(n, '_op', ''):
            op_node = uid + n._op
            dot.node(name=op_node, label=n._op)
            dot.edge(op_node, uid)

    # Connect children to the op node that produced their parent
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + getattr(n2, '_op', ''))

    return dot

In [112]:
import random

In [237]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
        # params = []
        # for neuron in self.neurons:
        #     params.extend(neuron.parameters())
        # return params


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [320]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])

In [321]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

# these are the expected targets
ys = [1.0, -1.0, -1.0, 1.0]

In [317]:
# loss = sum([(yout-ygt)**2 for ygt, yout in zip(ys, y_pred)])

In [316]:
# loss

In [315]:
# y_pred = [n(x) for x in xs]
# loss = sum([(yout-ygt)**2 for ygt, yout in zip(ys, y_pred)])
# loss

In [314]:
# loss.backward()

In [313]:
# for p in n.parameters():
#     p.data += -0.01 * p.grad 

In [312]:
# y_pred

In [346]:
for k in range(20):
    # forward pass
    y_pred = [n(x) for x in xs]
    loss = sum([(yout-ygt)**2 for ygt, yout in zip(ys, y_pred)])

    # zero grad
    for p in n.parameters():
        p.grad = 0.0
    
    # backward pass
    loss.backward()

    # change params
    for p in n.parameters():
        p.data += -0.01 * p.grad

    print("Iteration: ", k, "Loss: ", loss.data)

Iteration:  0 Loss:  0.0030604733642624576
Iteration:  1 Loss:  0.0030579578605319127
Iteration:  2 Loss:  0.003055446253731489
Iteration:  3 Loss:  0.0030529385350625453
Iteration:  4 Loss:  0.0030504346957524854
Iteration:  5 Loss:  0.0030479347270546374
Iteration:  6 Loss:  0.0030454386202481806
Iteration:  7 Loss:  0.0030429463666380436
Iteration:  8 Loss:  0.003040457957554832
Iteration:  9 Loss:  0.0030379733843546924
Iteration:  10 Loss:  0.0030354926384192626
Iteration:  11 Loss:  0.0030330157111555594
Iteration:  12 Loss:  0.0030305425939958683
Iteration:  13 Loss:  0.003028073278397681
Iteration:  14 Loss:  0.0030256077558435865
Iteration:  15 Loss:  0.003023146017841203
Iteration:  16 Loss:  0.003020688055923063
Iteration:  17 Loss:  0.0030182338616465145
Iteration:  18 Loss:  0.003015783426593651
Iteration:  19 Loss:  0.0030133367423712445


In [347]:
y_pred

[Value(data=0.9775504221446009),
 Value(data=-0.9674214935623944),
 Value(data=-0.9797356194215333),
 Value(data=0.96779209732765)]